In [ ]:
# !pip install transformers datasets sentencepiece sacrebleu accelerate pandas

In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/alexbenva/translated-output/translated_output.csv")

# Terms dataset
terms = df[["English Term","Tamil Term"]]
terms.columns = ["source","target"]

# Explanation dataset
exp = df[["english_description","Explanation"]]
exp.columns = ["source","target"]

# Combine both
final_dataset = pd.concat([terms,exp])

# remove empty rows
final_dataset = final_dataset.dropna()

# save training file
final_dataset.to_csv("nllb_training_dataset.csv",index=False)

print("Dataset size:",len(final_dataset))

Dataset size: 6599


In [ ]:
final_dataset = final_dataset.drop_duplicates()

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("nllb_training_dataset.csv")

dataset = Dataset.from_pandas(df)

dataset

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
tokenizer.src_lang = "eng_Latn"
target_lang = "tam_Taml"

In [ ]:
model.gradient_checkpointing_enable()

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def preprocess(example):

    inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True
    )

    targets = tokenizer(
        example["target"],
        max_length=64,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

tokenized_dataset = dataset.map(preprocess)

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/tamil_cyber_model",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    fp16=True,
    logging_steps=50,
    save_total_limit=2
)

In [ ]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator # Pass the data collator here
)

In [ ]:
.train()

In [ ]:
.save_model.save_model("/kaggle/working/tamil_cyber_model")
tokenizer.save_pretrained("/kaggle/working/tamil_cyber_model")
!zip -r tamil_cyber_model.zip /kaggle/working/tamil_cyber_model

In [4]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base_model = "facebook/nllb-200-distilled-600M"

model = AutoModelForSeq2SeqLM.from_pretrained(base_model)
tokenizer = AutoTokenizer.from_pretrained(base_model)

model = PeftModel.from_pretrained(
    model,
    "/kaggle/input/datasets/alexbenva/mymodel/kaggle/working/tamil_cyber1_model"
)

model = model.to("cuda")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [17]:
import re

def load_stopwords(path="/kaggle/input/datasets/alexbenva/stopwording/stopwords.txt"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return set(w.strip() for w in f if w.strip())
    except:
        return set()

STOPWORDS = load_stopwords()

TAMIL_RANGE = r"\u0B80-\u0BFF"

def join_tokens(text):

    tokens = text.split()
    fixed = []

    for tok in tokens:

        # attach suffix stopwords or single Tamil letters
        if fixed and (tok in STOPWORDS or re.fullmatch(f"[{TAMIL_RANGE}]", tok)):
            fixed[-1] = fixed[-1] + tok
        else:
            fixed.append(tok)

    return " ".join(fixed)


# -------------------------------
# Translation Function
# -------------------------------
def translate(text):

    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml"),
        max_new_tokens=80,
        num_beams=4
    )

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # join stopword suffixes
    translation = join_tokens(translation)

    # correction dictionary
    corrections = {
        "வடிவமைப்பு களை": "வடிவமைப்புகளை",
        "நெட்வொர்க்கையும்" : "இணையமைப்பையும்",
        "பூஜ்ஜிய நாள்":"வெற்று நாள்",
        "இணைய யமைப்பு": "இணையமைப்புகள்",
        "முறைமை களை" : "முறைமைகளை",
        "ஸ்பைவேர்":"உளவு மென்பொருள் ",
        "ஆஃப்லைன்" :"இணையமில்லா நிலையில் ",
        "நெட்வொர்க்குகள்" : "இணையமைப்பு",
        "டிஜிட்டல் ":"இலக்கமயம்",
        "மால்வேர் ":"தீய மென்பொருள்",
        "பாதிக்கியது":"பாதிக்கிறது"
    }

    for wrong, correct in corrections.items():
        translation = translation.replace(wrong, correct)

    return translation


In [57]:
print(translate("Always use a strong and unique password for your email account."))
print(translate("You should regularly update your software to patch security vulnerabilitiess"))
print(translate("Be cautious when clicking on links in unsolicited emails"))
print(translate("Lock your computer screen whenever you step away from your desk"))
print(translate("Network Security"))

உங்கள் மின்னஞ்சல் கணக்கிற்கு எப்போதும் ஒரு வலுவான மற்றும் தனித்துவமான கடவுச்சொல்லைப் பயன்படுத்தவும்.
பாதுகாப்பு பாதிப்புகளை சரிசெய்ய உங்கள் மென்பொருளை நீங்கள் தவறாமல் புதுப்பிக்க வேண்டும்
தேவையற்ற மின்னஞ்சல்களில் உள்ள இணைப்புகளை கிளிக் செய்யும் போது கவனமாக இருங்கள்
உங்கள் மேசையில்இருந்து விலகிச் செல்லும் ஒவ்வொரு முறையும் உங்கள் கணினி திரையை பூட்டுங்கள்
இணையமைப்புகள் பாதுகாப்பு


In [21]:
test_sentences = [
"Always use a strong and unique password for your email account.",
"Enabling two-factor authentication adds an extra layer of security.",
"You should regularly update your software to patch security vulnerabilities.",
"Be cautious when clicking on links in unsolicited emails.",
"It is important to back up your data regularly to prevent loss from ransomware.",
"Avoid using public Wi-Fi for online banking without a Virtual Private Network.",
"Lock your computer screen whenever you step away from your desk.",
"Social engineering attacks manipulate people into giving up confidential information.",

"The malware infected the entire network, causing widespread disruption.",
"Ransomware encrypted all the company's files and demanded a payment.",
"A keylogger can record every keystroke you type, including passwords.",
"The website was compromised and started distributing malware to visitors.",
"A zero-day exploit is a vulnerability that is unknown to the software vendor.",
"The Trojan horse disguised itself as a legitimate software update.",
"Spyware was secretly collecting the user's browsing history.",
"They launched a Distributed Denial of Service attack to take the server offline.",

"The system administrator configured the firewall to block all unauthorized access.",
"An Intrusion Detection System monitors network traffic for suspicious activity.",
"They use end-to-end encryption to protect messages from being intercepted.",
"The company's network was breached through a vulnerable port.",
"A Virtual Private Network creates a secure tunnel for your data.",
"The security team is performing a vulnerability scan to identify weak points.",
"They implemented network segmentation to isolate the critical systems.",
"The penetration test simulated an attack to find security holes.",

"Personally Identifiable Information must be handled with care.",
"The data breach exposed the personal details of millions of users.",
"Data encryption ensures that even if data is stolen, it cannot be read.",
"It is a common practice to hash passwords before storing them in a database.",
"The company's privacy policy explains how your data is collected and used.",
"Anonymization techniques can help protect user privacy in datasets.",
"They implemented multi-factor authentication to protect sensitive data.",
"A digital certificate helps verify the identity of a website.",

"The security team is currently investigating the security incident.",
"They created a forensic image of the hard drive for analysis.",
"The first step in incident response is to contain the breach.",
"Log analysis can help determine how the attacker gained access.",
"They traced the attack back to a specific IP address.",
"The company had to wipe the infected systems and restore from backups.",
"A rootkit was discovered hiding deep within the operating system.",
"The Chief Information Security Officer briefed the CEO on the data breach."
]

In [22]:
for sentence in test_sentences:
    print("EN:", sentence)
    print("TA:", translate(sentence))
    print("-"*80)

EN: Always use a strong and unique password for your email account.
TA: உங்கள் மின்னஞ்சல் கணக்கிற்கு எப்போதும் ஒரு வலுவான மற்றும் தனித்துவமான கடவுச்சொல்லைப் பயன்படுத்தவும்.
--------------------------------------------------------------------------------
EN: Enabling two-factor authentication adds an extra layer of security.
TA: இரண்டு காரணி அங்கீகாரத்தை செயல்படுத்துவது ஒரு கூடுதல் பாதுகாப்பு அடுக்கு சேர்க்கிறது.
--------------------------------------------------------------------------------
EN: You should regularly update your software to patch security vulnerabilities.
TA: பாதுகாப்பு குறைபாடுகளை சரிசெய்ய உங்கள் மென்பொருளை நீங்கள் தொடர்ந்து புதுப்பித்துக்கொள்ள வேண்டும்.
--------------------------------------------------------------------------------
EN: Be cautious when clicking on links in unsolicited emails.
TA: தேவையற்ற மின்னஞ்சல்களில் உள்ள இணைப்புகளை கிளிக் செய்யும் போது கவனமாக இருங்கள்.
--------------------------------------------------------------------------------
EN: It is imp

In [25]:
!pip install sacrebleu

In [ ]:
!pip install unbabel-comet

In [ ]:
!pip install transformers==4.30.2
!pip install unbabel-comet==2.2.2
!pip install sentencepiece sacrebleu

In [ ]:
!pip uninstall -y tensorflow tensorflow-text tf-keras keras
!pip uninstall -y transformers
!pip uninstall -y unbabel-comet

In [59]:
import sacrebleu

references = [
"உங்கள் மின்னஞ்சல் கணக்கிற்கு எப்போதும் ஒரு வலுவான மற்றும் தனித்துவமான கடவுச்சொல்லைப் பயன்படுத்தவும்.",
"பாதுகாப்பு பாதிப்புகளை சரிசெய்ய உங்கள் மென்பொருளை நீங்கள் தவறாமல் புதுப்பிக்க வேண்டும்",
"நீங்கள் இணையத்தளத்தில் பொருட்களை நுகர விருப்பப்பட்டால் அதற்கு முன் உங்கள் கணினியின் அனைத்துப் பாதுகாப்பு அம்சங்களையும் சரிப்பார்க்கவும் அதாவது வைர அற்றதாக உள்ளதா என்று சரிபார்க்கவும்",
"மீட்பு மென்பொருளால் ஏற்படும் இழப்பைத் தடுக்க உங்கள் தரவுகளை தவறாமல் காப்புப் பிரதி எடுப்பது முக்கியம்",
"இந்த தீய மென்பொருள்முழு இணையமைப்பையும் பாதித்து, பரவலான இடையூறுகளை ஏற்படுத்தியது"    
]

predictions = [
translate("The malware infected the entire network, causing widespread disruption."),
translate("You should regularly update your software to patch security vulnerabilitiess"),
translate("Before you go for online shopping make sure your PC is secured with all core protections like an antivirus, anti spyware,	firewall,	system	updated	with	all	patches	and	web	browser security with the trusted sites and security level at high."),
translate("It is important to back up your data regularly to prevent loss from ransomware"),
translate("The malware infected the entire network, causing widespread disruption")
]

bleu = sacrebleu.corpus_bleu(predictions, [references])

print("BLEU score:", bleu.score)
chrf = sacrebleu.corpus_chrf(predictions, [references])

print("chrF score:", chrf.score)
ter = sacrebleu.corpus_ter(predictions, [references])

print("TER score:", ter.score)



BLEU score: 28.725276598607348
chrF score: 61.88065283808013
TER score: 81.03448275862068


In [6]:
from comet import download_model, load_from_checkpoint

# Source sentences
sources = [
"The malware infected the entire network, causing widespread disruption.",
"You should regularly update your software to patch security vulnerabilities",
"Before you go for online shopping make sure your PC is secured with all core protections like antivirus and firewall.",
"Before you buy things online research about the website and verify the vendor details."
]

# Reference translations
references = [
"மால்வேர் முழு நெட்வொர்க்கையும் பாதித்து பெரிய குழப்பத்தை ஏற்படுத்தியது.",
"பாதுகாப்பு பாதிப்புகளை சரிசெய்ய உங்கள் மென்பொருளை தவறாமல் புதுப்பிக்க வேண்டும்.",
"ஆன்லைன் வாங்குவதற்கு முன் உங்கள் கணினியில் வைரஸ் எதிர்ப்பு மற்றும் பாதுகாப்பு அமைப்புகள் உள்ளதா என்பதை சரிபார்க்கவும்.",
"ஆன்லைனில் பொருட்கள் வாங்குவதற்கு முன் அந்த இணையதளத்தை ஆராய்ந்து விற்பனையாளர் விவரங்களை சரிபார்க்கவும்."
]

# Your model predictions
predictions = [
translate(s) for s in sources
]

# Format for COMET
data = [
    {"src": s, "mt": p, "ref": r}
    for s, p, r in zip(sources, predictions, references)
]

# Download model
model_path = download_model("Unbabel/wmt22-comet-da")

# Load model
model = load_from_checkpoint(model_path)

# Evaluate
model_output = model.predict(data, batch_size=4, gpus=0)

print("COMET Score:", model_output.system_score)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automati

COMET Score: 0.9269914627075195
